In [1]:
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)

import numpy as np
import pandas as pd
from common.const import PRODUCT_VECTOR_DB_COLLECTION, DEVICE, SCHEMA, TABLE_RAW_PRODUCTS, TABLE_RAW_REVIEWS, TABLE_FEATURED_DENSE_VECTORS
from common.helper import connect_clickhouse
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import MinMaxScaler
import faiss
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from common.const import SCHEMA, TABLE_RAW_PRODUCTS, DEVICE
from common.helper import clickhouse_extract_data_as_df
import clickhouse_connect
import joblib


c:\Users\Admin\.conda\envs\scc454\lib\site-packages\torchvision\io\image.py:13: UserWarning: Failed to load image Python extension: '[WinError 127] The specified procedure could not be found'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


In [2]:
# Query data

def query_products_data(n_rows, offset):
    products_query = f"""
    SELECT * FROM {SCHEMA}.{TABLE_RAW_PRODUCTS}
    LIMIT {n_rows} OFFSET {offset}
    """
    return clickhouse_extract_data_as_df(products_query)

def query_reviews_data(n_rows, offset):
    reviews_query = f"""
    SELECT * FROM {SCHEMA}.{TABLE_RAW_REVIEWS} LIMIT {n_rows} OFFSET {offset}
    """
    return clickhouse_extract_data_as_df(reviews_query)

In [3]:
#high signal vectoring: short but important field like main_category, title, store, categories, product_category
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
def create_high_signal_vectoring(row):
    text_parts = []

    if pd.notna(row['main_category']):
        text_parts.append(row['main_category'])

    if pd.notna(row['title']):
        text_parts.append(row['title'])

    if pd.notna(row['store']):
        text_parts.append(row['store'])

    if pd.notna(row['categories']):
        text_parts.append(row['categories'])

    if pd.notna(row['product_category']):
        text_parts.append(row['product_category'])

    return ' '.join(text_parts)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
def create_low_signal_vectoring(row): #features, description, details
    text_parts = []

    if pd.notna(row['features']):
        text_parts.append(row['features'])

    if pd.notna(row['description']):
        text_parts.append(row['description'])

    if pd.notna(row['details']):
        text_parts.append(row['details'])

    return ' '.join(text_parts)


In [5]:
# query_drop = f"DROP TABLE IF EXISTS {SCHEMA}.{TABLE_FEATURED_DENSE_VECTORS}"
query = f"""CREATE TABLE IF NOT EXISTS {SCHEMA}.{TABLE_FEATURED_DENSE_VECTORS} (
    id UInt64,
    parent_asin String,
    vector_high_signal Array(Float32),
    vector_low_signal Array(Float32),
    vector_high_low_signal Array(Float32),
    mixed_vector Array(Float32),
    vector_tfidf Array(Float32),
    vector_hashed Array(Float32),
    event_time DateTime DEFAULT now()
) ENGINE = ReplacingMergeTree(event_time)
ORDER BY parent_asin
PRIMARY KEY parent_asin;""" #use replacing merge tree to keep only the latest vector for each product, in case of updates
client = connect_clickhouse()
# client.command(query_drop)
# client.command(query)
client.close()

In [17]:
# 1. Setup Models and Scalers
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
vectorizer = TfidfVectorizer(
    max_features=1000,
    min_df=2,
    max_df=0.8,
    ngram_range=(1, 2),
    stop_words='english'
)
# train_df = query_products_data(10000, 0) # Use a sample to fit the vectorizer
# corpus = train_df.apply(create_high_signal_vectoring, axis=1) + \
#          " " + train_df.apply(create_low_signal_vectoring, axis=1)
# vectorizer.fit(corpus)
# joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

scaler = MinMaxScaler()

# New: Hashing Vectorizer (Stateless, no fit needed)
hashing_vec = HashingVectorizer(
    n_features=1024, 
    stop_words='english', 
    alternate_sign=False, # Keeps values positive like TF-IDF
    norm='l2'
)

def create_tfidf_vectors(high_text, low_text, vectorizer):
    # Combine everything: Title, Category, Features, and Description
    # This ensures the 'Vocabulary' of the vectorizer sees every possible keyword
    # Space-join them so words don't run together
    full_corpus = high_text + " " + low_text
    tfidf_matrix = vectorizer.transform(full_corpus)
        
    vectors_dense = tfidf_matrix.toarray().astype('float32')
    vectors_normalized = normalize(vectors_dense, norm='l2')
    
    return vectors_normalized


# We "fit" the scaler on a sample or pre-defined bounds to ensure 
# batch 1 and batch 100 use the same scale.
# Expected ranges: Rating(0-5), Price(0-500?), Count(0-10000?)
# Or: fit it on your first large batch.

def get_existing_asins():
    """Get all existing parent_asin as set (fast lookup)."""
    query = f"SELECT parent_asin FROM {SCHEMA}.{TABLE_FEATURED_DENSE_VECTORS}"
    df = clickhouse_extract_data_as_df(query)
    return set(df['parent_asin'].astype(str).tolist())

def get_remaining_asins() -> list[str]:
    """Return all parent_asins in TABLE_RAW_PRODUCTS not yet in TABLE_FEATURED_DENSE_VECTORS."""
    query = f"""
        SELECT parent_asin
        FROM {SCHEMA}.{TABLE_RAW_PRODUCTS}
        WHERE parent_asin NOT IN (
            SELECT parent_asin FROM {SCHEMA}.{TABLE_FEATURED_DENSE_VECTORS}
        )
    """
    df = clickhouse_extract_data_as_df(query)
    return df['parent_asin'].astype(str).tolist()

def query_products_by_asins(client, asins: list[str]) -> pd.DataFrame:
    """Fetch raw product rows using a temp table to avoid IN-list query size limits."""
    
    # 1. Create a temporary table for the ASIN lookup
    client.command(f"""
        CREATE TEMPORARY TABLE temp_asin_lookup (parent_asin String)
    """)
    
    # 2. Insert ASINs into the temp table in chunks to avoid insert limits too
    chunk_size = 5000
    for i in range(0, len(asins), chunk_size):
        chunk = asins[i : i + chunk_size]
        client.insert("temp_asin_lookup", 
                      [[a] for a in chunk], 
                      column_names=["parent_asin"])
    
    # 3. JOIN against temp table instead of inline IN list
    query = f"""
        SELECT p.* 
        FROM {SCHEMA}.{TABLE_RAW_PRODUCTS} AS p
        INNER JOIN temp_asin_lookup AS t USING (parent_asin)
    """
    df = client.query_df(query)
    
    # 4. Temp table is session-scoped, auto-dropped on connection close
    #    but explicit cleanup is good practice
    client.command("DROP TEMPORARY TABLE IF EXISTS temp_asin_lookup")
    
    return df


def manual_scale(val, min_v, max_v):
        return (np.clip(val, min_v, max_v) - min_v) / (max_v - min_v)

def process_streaming_to_clickhouse(total_rows, batch_size=1000):
    vectorizer = joblib.load("tfidf_vectorizer.pkl")
    client = connect_clickhouse()
    # 1. Get only unprocessed ASINs upfront
    print("Fetching remaining ASINs...")
    remaining_asins = get_remaining_asins()
    total_remaining = len(remaining_asins)
    print(f"{total_remaining:,} ASINs left to vectorise.")

    if total_remaining == 0:
        print("Nothing to do — all ASINs are already vectorised.")
        client.close()
        return

    # 2. Stable ID baseline
    max_id_result  = client.query(f"SELECT max(id) FROM {SCHEMA}.{TABLE_FEATURED_DENSE_VECTORS}")
    current_max_id = max_id_result.result_rows[0][0] or 0
    next_id        = current_max_id + 1
    print(f"Next row ID starts at {next_id:,}")

    inserted_count = 0

    for chunk_start in range(0, total_remaining, batch_size):
        # 2. Extract only the current chunk
        asin_chunk = remaining_asins[chunk_start : chunk_start + batch_size]
        df_batch = query_products_by_asins(client, asin_chunk)

        if df_batch.empty:
            print(f"Chunk {chunk_start}: no rows returned, skipping.")
            continue

        print(f"Chunk {chunk_start}: processing {len(df_batch)} / {total_remaining:,} remaining...")

        
        # 3. Clean and Scale Numerical data per batch
        df_batch['price'] = pd.to_numeric(df_batch['price'], errors='coerce').fillna(20.0)
        df_batch['average_rating'] = pd.to_numeric(df_batch['average_rating'], errors='coerce').fillna(3.0)
        df_batch['rating_number'] = pd.to_numeric(df_batch['rating_number'], errors='coerce').fillna(0)
        
        # Apply scaling (consistent bounds)
        s_rating = manual_scale(df_batch['average_rating'].values, 0, 5)
        s_count = manual_scale(df_batch['rating_number'].values, 0, 10000)
        s_price = manual_scale(df_batch['price'].values, 0, 500)
        batch_nums = np.column_stack([s_rating, s_count, s_price])

        # 4. Vector Encoding
        high_text = df_batch.apply(create_high_signal_vectoring, axis=1)
        low_text = df_batch.apply(create_low_signal_vectoring, axis=1)
        high_vecs = model.encode(high_text.tolist(), 
                                 normalize_embeddings=True)
        low_vecs = model.encode(low_text.tolist(), 
                                normalize_embeddings=True)
        
        tfidf_vecs = create_tfidf_vectors(high_text, low_text, vectorizer)
        hashed_vecs = hashing_vec.transform(high_text + " " + low_text).toarray().astype('float32')
        
        # 5. Create Mixed Vector
        text_mean = (high_vecs + low_vecs) / 2
        weighted_nums = batch_nums * 3.0 # Give more weight to numerical features in the mixed vector
        mixed_vecs = np.hstack([text_mean, weighted_nums])
        mixed_vec = mixed_vecs / np.linalg.norm(mixed_vecs, axis=1, keepdims=True)
        # 6. Prepare and Insert
        data_to_insert = []
        for idx in range(len(df_batch)):
            data_to_insert.append((
                int(next_id + inserted_count + idx),                               
                str(df_batch.iloc[idx]['parent_asin']),        
                high_vecs[idx].tolist(),                    
                low_vecs[idx].tolist(),
                text_mean[idx].tolist(),
                mixed_vec[idx].tolist(),                    
                tfidf_vecs[idx].tolist(),
                hashed_vecs[idx].tolist(),
            ))
        if not data_to_insert:
            print(f"Chunk {chunk_start}: data_to_insert is empty, skipping insert.")
            continue

        
        client.insert(f"{SCHEMA}.{TABLE_FEATURED_DENSE_VECTORS}", data_to_insert, 
                      column_names=['id', 'parent_asin', 'vector_high_signal', 'vector_low_signal', 
                                    'vector_high_low_signal', 'mixed_vector', 'vector_tfidf', 'vector_hashed'])
        
        print(f"Processed chunk {chunk_start}...")
        
    client.close()

# Start the stream for 50k rows
process_streaming_to_clickhouse(total_rows=50000000, batch_size=10000)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Fetching remaining ASINs...
connected to clickhouse
280,737 ASINs left to vectorise.
Next row ID starts at 22,275,766
Chunk 0: processing 10000 / 280,737 remaining...
Processed chunk 0...
Chunk 10000: processing 10000 / 280,737 remaining...
Processed chunk 10000...
Chunk 20000: processing 10000 / 280,737 remaining...
Processed chunk 20000...
Chunk 30000: processing 10000 / 280,737 remaining...
Processed chunk 30000...
Chunk 40000: processing 10000 / 280,737 remaining...
Processed chunk 40000...
Chunk 50000: processing 10000 / 280,737 remaining...
Processed chunk 50000...
Chunk 60000: processing 10000 / 280,737 remaining...
Processed chunk 60000...
Chunk 70000: processing 10000 / 280,737 remaining...
Processed chunk 70000...
Chunk 80000: processing 10000 / 280,737 remaining...
Processed chunk 80000...
Chunk 90000: processing 10000 / 280,737 remaining...
Processed chunk 90000...
Chunk 100000: processing 10000 / 280,737 remaining...
Processed chunk 100000...
Chunk 110000: processing 10000